In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, pearsonr

# Варіант 1 (Київ)
np.random.seed(1)
city = "Київ"
monthly_temp = [-4, -3, 2, 9, 15, 19, 21, 20, 14, 8, 2, -2]   # 12 значень для Києва
months = ["Січ", "Лют", "Бер", "Кві", "Tра", "Чер",
          "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

rows = []
for month, temp in zip(months, monthly_temp):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": city, "місяць": month, "температура": temp,
        "опалювальні_дні": heating_days, "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)
print(climate)

In [ ]:
X = climate["температура"]
Y = climate["витрати_на_опалення"]

result = linregress(X, Y)
print(f"slope (b1) = {result.slope:.3f}")
print(f"intercept (b0) = {result.intercept:.3f}")
print(f"R² = {result.rvalue ** 2:.3f}")
print(f"p-value = {result.pvalue:.4e}")

# Випишемо рівняння регресії
print(f"Рівняння регресії: витрати ≈ {result.intercept:.3f} + {result.slope:.3f} × температура")

In [ ]:
# Спосіб 1: через результат linregress (rvalue ** 2)
r2_method1 = result.rvalue ** 2

# Спосіб 2: через суми квадратів (SS_res / SS_tot)
predicted = result.intercept + result.slope * X
residuals = Y - predicted

ss_residuals = np.sum(residuals ** 2)
ss_total = np.sum((Y - np.mean(Y)) ** 2)
r2_method2 = 1 - (ss_residuals / ss_total)

print(f"R² (через rvalue): {r2_method1:.4f}")
print(f"R² (через формулу 1 - SS_res/SS_tot): {r2_method2:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Графік 1: Регресійна пряма
ax1.scatter(X, Y, color="blue", label="Фактичні дані")
ax1.plot(X, predicted, color="red", label="Регресійна пряма")
ax1.set_xlabel("Середньомісячна температура, °C")
ax1.set_ylabel("Витрати на опалення")
ax1.set_title("Лінійна регресія")
ax1.legend()
ax1.grid(True)

# Графік 2: Графік залишків
ax2.scatter(X, residuals, color="purple", label="Залишки")
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_xlabel("Середньомісячна температура, °C")
ax2.set_ylabel("Залишок (Y - Y_predicted)")
ax2.set_title("Графік залишків")
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Прогноз для температури -10°C
x_new = -10
y_pred_new = result.intercept + result.slope * x_new

print(f"Прогнозовані витрати для температури {x_new}°C: {y_pred_new:.3f}")
print(f"Фактичний діапазон температур у наборі: від {X.min()}°C до {X.max()}°C")

In [ ]:
p_val_reg = result.pvalue
p_val_corr = pearsonr(X, Y).pvalue

print(f"p-value з linregress: {p_val_reg:.6e}")
print(f"p-value з pearsonr: {p_val_corr:.6e}")

### Відповіді до Завдання 5 та Контрольні запитання:

#### Чому p-value з linregress та pearsonr однакові?
Для простої лінійної регресії з одним предиктором нульова гіпотеза про те, що нахил регресії дорівнює нулю ($H_0: b_1 = 0$), математично еквівалентна нульовій гіпотезі про те, що коефіцієнт кореляції дорівнює нулю ($H_0: r = 0$). Тому статистичні тести дають ідентичні рівні значущості ($p$-value).

---

### Відповіді на контрольні питання:

1. **Чому метод найменших квадратів мінімізує суму квадратів залишків, а не суму самих залишків чи їхніх модулів?**
   Підсумовування простих залишків неможливе, оскільки додатні й від'ємні відхилення взаємно компенсували б одне одного (сума залишків у регресії зазвичай дорівнює нулю). Модулі (МНК-абсолютне відхилення) важко диференціювати аналітично через злам у нулі. Метод найменших квадратів (МНК) використовує квадрати, що дає змогу легко знайти єдине аналітичне рішення через похідні, а також сильніше штрафує великі викиди й помилки.

2. **Що означає коефіцієнт нахилу $b_1$ змістовно і чим відрізняється від коефіцієнта кореляції $r$?**
   * Нахил $b_1$ вимірюється в **абсолютних одиницях вимірювання змінних** (наприклад, тис. грн на $1^\circ\text{C}$) і показує швидкість зміни залежної змінної при зміні незалежної на одиницю.
   * Коефіцієнт кореляції $r$ є **безрозмірною величиною** від -1 до 1, яка характеризує лише тісноту та напрям лінійного зв'язку, але не масштаб змін.

3. **Чому високий $R^2$ сам по собі не є достатньою підставою вважати лінійну модель адекватною?**
   Високе значення $R^2$ свідчить лише про загальну тісноту зв'язку, проте воно не показує наявність систематичних помилок. Графік залишків допомагає виявити приховану нелінійність, гетероскедастичність (зміну розкиду залишків) або вплив специфічних обмежень (як-от обнулення витрат), які роблять лінійну модель формально хибною.

4. **Що змінюється в надійності прогнозу, коли значення $x$ виходить за межі спостереженого діапазону даних?**
   Надійність різко паждає, оскільки модель за межами вибірки спирається лише на припущення про збереження тенденції (екстраполяція). Будь-які непередбачувані нелінійні ефекти поза межами зібраних даних роблять прогноз хибним.